In [1]:
import xarray as xr
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, ConvLSTM2D, Conv2D, BatchNormalization, TimeDistributed, Lambda
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K



2025-09-24 18:29:37.836391: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-24 18:29:37.947645: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-24 18:29:37.976260: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-24 18:29:38.184581: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-24 18:29:39.539271: W tensorflow/comp

In [154]:
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except:
        pass

tf.keras.mixed_precision.set_global_policy('mixed_float16')

NC_PATH   = "data/SST_GLO_L4/raw/ostia_sst_1996-2022_clean.nc"
LOOKBACK  = 30
HORIZONS  = (1,2,3,4,5,6,7)
MIN_VALID = 0.98
TRAIN = slice("1996-06-01", "2016-12-31")
VAL   = slice("2017-01-01", "2019-12-31")
TEST  = slice("2020-01-01", "2022-05-31")
BATCH_SIZE = 4

In [155]:
def build_ocean_mask_bitmask(ds: xr.Dataset, mode="first", ice_threshold=0.0) -> np.ndarray:
    m = ds["mask"].astype("int16")
    is_water_t = (m & 1) != 0
    is_ice_t   = (m & 8) != 0
    if mode == "first":
        is_water = is_water_t.isel(time=0)
        is_ice   = is_ice_t.isel(time=0)
    elif mode == "majority":
        is_water = (is_water_t.mean("time") > 0.5)
        is_ice   = (is_ice_t.mean("time") > 0.5)
    else:
        raise ValueError("mode deve ser 'first' ou 'majority'")
    ocean_mask = is_water & (~is_ice)
    if ("sea_ice_fraction" in ds) and (ice_threshold is not None):
        sif = ds["sea_ice_fraction"]
        if float(sif.max().compute()) > 1.01:
            sif = sif / 100.0
        sif2d = sif.isel(time=0) if mode == "first" else sif.mean("time")
        ocean_mask = ocean_mask & (sif2d <= ice_threshold)
    return ocean_mask.values

In [156]:
def normalize_only_ocean(da: xr.DataArray, ocean_mask: np.ndarray):
    mu = da.where(ocean_mask).mean("time").values.astype(np.float32)
    sigma = da.where(ocean_mask).std("time").values.astype(np.float32)
    sigma = np.where((sigma < 1e-6) | ~np.isfinite(sigma), 1.0, sigma)
    arr = da.values
    arr_n = (arr - mu) / sigma
    arr_n[:, ~ocean_mask] = np.nan
    return xr.DataArray(arr_n, coords=da.coords, dims=da.dims), mu, sigma


In [157]:
ds = xr.open_dataset(NC_PATH, decode_cf=True)
sst = ds["sst"]

ocean_mask = build_ocean_mask_bitmask(ds, mode="first", ice_threshold=0.0)  # (H,W)
H, W = int(ocean_mask.shape[0]), int(ocean_mask.shape[1])

sst_tr = sst.sel(time=TRAIN)
sst_va = sst.sel(time=VAL)

sst_tr_n, mu, sigma = normalize_only_ocean(sst_tr, ocean_mask)
arr_va = (sst_va.values - mu) / sigma
arr_va[:, ~ocean_mask] = np.nan
sst_va_n = xr.DataArray(arr_va, coords=sst_va.coords, dims=sst_va.dims)

sst_tr_n_tf = tf.constant(sst_tr_n.values, dtype=tf.float32)  # (T,H,W)
sst_va_n_tf = tf.constant(sst_va_n.values, dtype=tf.float32)  # (T,H,W)

max_h = max(HORIZONS)
train_indices = np.arange(LOOKBACK, sst_tr_n.shape[0] - max_h + 1, dtype=np.int32)
val_indices   = np.arange(LOOKBACK, sst_va_n.shape[0] - max_h + 1, dtype=np.int32)
train_ds_indices = tf.data.Dataset.from_tensor_slices(train_indices)
val_ds_indices   = tf.data.Dataset.from_tensor_slices(val_indices)


/home/henrique/anaconda3/envs/sstml/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [158]:
# X (L,H,W,1), Y (H,W,Hn)
def map_fn(index):
    index = tf.cast(index, tf.int32)
    x = sst_tr_n_tf[index-LOOKBACK:index, :, :]         # (L,H,W)
    x = tf.expand_dims(x, -1)                           # (L,H,W,1)
    y_stack = tf.gather(sst_tr_n_tf, index + H_T - 1)   # (Hn,H,W)
    y = tf.transpose(y_stack, [1, 2, 0])                # (H,W,Hn)
    return x, y

def map_fn_val(index):
    index = tf.cast(index, tf.int32)
    x = sst_va_n_tf[index-LOOKBACK:index, :, :]
    x = tf.expand_dims(x, -1)
    y_stack = tf.gather(sst_va_n_tf, index + H_T - 1)
    y = tf.transpose(y_stack, [1, 2, 0])
    return x, y

In [159]:

# Filtro MIN_VALID no oceano
ocean_bool = tf.constant(ocean_mask, dtype=tf.bool)     # (H,W)
ocean4_x   = tf.reshape(ocean_bool, (1, H, W, 1))       # para X (tile em LOOKBACK)
ocean_hw1  = tf.reshape(ocean_bool, (H, W, 1))          # para Y (broadcast no Hn)
NUM_OCEAN  = tf.constant(float(np.sum(ocean_mask)), tf.float32)

# Filtro: mantém apenas janelas com proporção mínima de válidos no oceano
def valid_filter(x, y):
    # x: (L,H,W,1), y: (H,W,Hn)
    x_fin = tf.math.is_finite(x)
    y_fin = tf.math.is_finite(y)

    x_m = tf.logical_and(x_fin, tf.tile(ocean4_x, [LOOKBACK, 1, 1, 1]))
    y_m = tf.logical_and(y_fin, ocean_hw1)              # (H,W,1) ~> broadcast para (H,W,Hn)

    x_cnt = tf.reduce_sum(tf.cast(x_m, tf.float32))
    y_cnt = tf.reduce_sum(tf.cast(y_m, tf.float32))
    total_possible = (LOOKBACK + len(HORIZONS)) * NUM_OCEAN
    ratio = (x_cnt + y_cnt) / (total_possible + 1e-12)
    return ratio >= tf.constant(MIN_VALID, tf.float32)

In [160]:
def sanitize_batch(x, y):
    x = tf.where(tf.math.is_finite(x), x, 0.0)
    return x, y


In [161]:
BATCH_SIZE = 8

train_dataset = (train_ds_indices
    .shuffle(len(train_indices))
    .map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .filter(valid_filter)
    .batch(BATCH_SIZE)
    .map(sanitize_batch, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE))

val_dataset = (val_ds_indices
    .map(map_fn_val, num_parallel_calls=tf.data.AUTOTUNE)
    .filter(valid_filter)
    .batch(BATCH_SIZE)
    .map(sanitize_batch, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .prefetch(tf.data.AUTOTUNE))

# ----------------------------------

In [162]:
# Máscara 4D para oceano
ocean_mask_tf = tf.constant(ocean_mask.astype("float32"))
ocean_mask_tf = tf.reshape(ocean_mask_tf, (1, H, W, 1))   # (1,H,W,1) -> broadcast em Hn

def masked_mse(y_true, y_pred):
    # y_*: (B,H,W,Hn)
    y_pred = tf.cast(y_pred, tf.float32)
    finite = tf.math.is_finite(y_true)                    # (B,H,W,Hn)
    mask = tf.cast(finite, tf.float32) * ocean_mask_tf    # (1,H,W,1) -> (B,H,W,Hn)
    diff = y_pred - tf.where(finite, y_true, 0.0)
    se = tf.square(diff)
    return tf.reduce_sum(se * mask) / (tf.reduce_sum(mask) + K.epsilon())


In [163]:
from tensorflow.keras.layers import Input, ConvLSTM2D, Conv2D, BatchNormalization
from tensorflow.keras import Model

def build_convlstm_fast(input_shape, n_horizons, base_filters=24):
    inp = Input(shape=input_shape)  # (L,H,W,1)
    x = ConvLSTM2D(base_filters, (3,3), padding='same', return_sequences=True)(inp)
    x = BatchNormalization()(x)
    x = ConvLSTM2D(base_filters, (3,3), padding='same', return_sequences=False)(x)
    x = BatchNormalization()(x)
    # Saída direta por pixel com Hn canais
    out = Conv2D(n_horizons, 1, padding='same', activation=None, dtype=tf.float32)(x)  # (B,H,W,Hn)
    return Model(inp, out)

input_shape = (LOOKBACK, H, W, 1)
n_horizons = len(HORIZONS)
model = build_convlstm_fast(input_shape, n_horizons, base_filters=24)  # 24 filtros ajuda a caber
model.compile(
    optimizer=Adam(learning_rate=3e-4, clipnorm=1.0),
    loss=masked_mse,
    # jit_compile=False  # padrão; deixe explícito se tinha setado True antes
)


In [164]:
print("\nChecando um batch...")
for xb, yb in train_dataset.take(1):
    print("X:", xb.shape, "| Y:", yb.shape)
    break

print("\nTreinando...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    # passos menores se quiser validar rápido:
    # steps_per_epoch=300, validation_steps=80,
)



Checando um batch...
X: (8, 30, 100, 100, 1) | Y: (8, 100, 100, 7)

Treinando...
Epoch 1/10


2025-09-19 18:38:21.527001: W external/local_tsl/tsl/framework/bfc_allocator.cc:482] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.00GiB (rounded to 2148642048)requested by op 
2025-09-19 18:38:21.527227: I external/local_tsl/tsl/framework/bfc_allocator.cc:1039] BFCAllocator dump for GPU_0_bfc
2025-09-19 18:38:21.527258: I external/local_tsl/tsl/framework/bfc_allocator.cc:1046] Bin (256): 	Total Chunks: 6673, Chunks in use: 6673. 1.63MiB allocated for chunks. 1.63MiB in use in bin. 36.2KiB client-requested in use in bin.
2025-09-19 18:38:21.527280: I external/local_tsl/tsl/framework/bfc_allocator.cc:1046] Bin (512): 	Total Chunks: 68, Chunks in use: 68. 35.2KiB allocated for chunks. 35.2KiB in use in bin. 33.6KiB client-requested in use in bin.
2025-09-19 18:38:21.527301: I external/local_tsl/tsl/framework/bfc_allocator.cc:1046] Bin (1024): 	Total Chunks: 22, Chunks in use: 22. 28.8KiB allocated for chunks. 28.8KiB in use in bin. 24.2KiB client-requested in use in bin.
2

ResourceExhaustedError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "/home/henrique/anaconda3/envs/sstml/lib/python3.10/runpy.py", line 196, in _run_module_as_main

  File "/home/henrique/anaconda3/envs/sstml/lib/python3.10/runpy.py", line 86, in _run_code

  File "/home/henrique/anaconda3/envs/sstml/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/henrique/anaconda3/envs/sstml/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/henrique/anaconda3/envs/sstml/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/henrique/anaconda3/envs/sstml/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/home/henrique/anaconda3/envs/sstml/lib/python3.10/asyncio/base_events.py", line 603, in run_forever

  File "/home/henrique/anaconda3/envs/sstml/lib/python3.10/asyncio/bas